In [1]:
import duckdb

con = duckdb.connect("f1.duckdb")

,name
0,circuits
1,constructor_results
2,constructor_standings
3,constructors
4,driver_season
5,driver_standings
6,drivers
7,lap_times
8,pit_stops
9,qualifying


In [2]:
con.close()

I forgot to populate the races entered column

In [4]:
con = duckdb.connect("f1.duckdb")

con.execute(
"UPDATE driver_season ds "
"SET races_entered = agg.races_entered "
"FROM ( "
"    SELECT driverId, year, "
"           COUNT(DISTINCT raceId) AS races_entered "
"    FROM results_drivers_races_circuits "
"    WHERE year >= 2014 "
"    GROUP BY driverId, year "
") agg "
"WHERE ds.driverId = agg.driverId "
"AND ds.year = agg.year "
)

con.execute("SELECT * FROM driver_season").df()

,driverId,year,total_points,races_entered,dnfs,avg_grid,avg_finish,completed_races,ref,constructor
0,3,2014,317.0,19,2,1.684211,2.529412,16,rosberg,131
1,825,2014,55.0,19,1,8.789474,9.388889,12,kevin_magnussen,1
2,18,2014,126.0,19,1,8.473684,7.500000,13,button,1
3,4,2014,161.0,19,2,6.526316,5.411765,17,alonso,6
4,822,2014,186.0,19,1,6.210526,5.555556,17,bottas,3
...,...,...,...,...,...,...,...,...,...,...
239,154,2020,2.0,15,3,14.133333,14.666667,6,grosjean,210
240,20,2020,33.0,17,2,12.058824,10.400000,9,vettel,6
241,857,2023,82.0,22,3,9.636364,9.631579,14,piastri,1
242,840,2024,24.0,13,1,11.615385,11.166667,8,stroll,117


In [5]:
con.close()

## Driver reliability

- DNF rate
- Position delta
- High altitude success rate

These features main purpose is to represent the factors that come into play that are influeced by a driver, it is expected to have an overlap with non-driver factors.

Starting i will create a dnf rate feature, this indicates how many races in a season a driver does not finish. This is one of the most basic explanatory features

### DNF rate

In [6]:
con = duckdb.connect("f1.duckdb")

con.execute("CREATE TABLE features AS "
"SELECT "
    "ds.driverId, "
    "ds.year, "
    "SUM(CASE WHEN r.milliseconds IS NULL THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS DNF_rate "
" FROM driver_season ds "
"JOIN results_drivers_races_circuits r "
    "ON ds.driverId = r.driverId AND ds.year = r.year "
"GROUP BY ds.driverId, ds.year;")

In [7]:
con.execute("DESCRIBE driver_season").df()

,column_name,column_type,null,key,default,extra
0,driverId,BIGINT,YES,NaN,NaN,NaN
1,year,BIGINT,YES,NaN,NaN,NaN
2,total_points,DOUBLE,YES,NaN,NaN,NaN
3,races_entered,BIGINT,YES,NaN,NaN,NaN
4,dnfs,BIGINT,YES,NaN,NaN,NaN
5,avg_grid,DOUBLE,YES,NaN,NaN,NaN
6,avg_finish,DOUBLE,YES,NaN,NaN,NaN
7,completed_races,BIGINT,YES,NaN,NaN,NaN
8,constructor,VARCHAR,YES,NaN,NaN,NaN
9,ref,VARCHAR,YES,NaN,NaN,NaN


In [8]:
con.close()

In [9]:
con = duckdb.connect("f1.duckdb")
con.execute("ALTER TABLE features "
"ADD COLUMN ref VARCHAR;")

In [10]:
con.execute("ALTER TABLE features "
"ADD COLUMN constructor VARCHAR;")

In [11]:
con.execute("UPDATE features f "
"SET ref = ds.ref "
"FROM driver_season ds "
"WHERE f.driverId = ds.driverId "
    "AND f.year = ds.year;")

In [12]:
con.execute("UPDATE features f "
"SET constructor = ds.constructor "
"FROM driver_season ds "
"WHERE f.driverId = ds.driverId "
    "AND f.year = ds.year;")


In [13]:
con.execute("ALTER TABLE features "
"ADD COLUMN constructorRef VARCHAR;")

In [14]:
con.execute("UPDATE features f "
"SET constructorRef = c.constructorRef "
"FROM constructors c "
"WHERE f.constructor = c.constructorId;")
con.close()

In [15]:
con = duckdb.connect("f1.duckdb")

con.execute("SELECT * FROM features").df()

,driverId,year,DNF_rate,ref,constructor,constructorRef
0,3,2014,0.157895,rosberg,131,mercedes
1,825,2014,0.368421,kevin_magnussen,1,mclaren
2,18,2014,0.315789,button,1,mclaren
3,4,2014,0.105263,alonso,6,ferrari
4,822,2014,0.105263,bottas,3,williams
...,...,...,...,...,...,...
239,859,2023,0.400000,lawson,213,alphatauri
240,20,2020,0.470588,vettel,6,ferrari
241,825,2023,0.681818,kevin_magnussen,210,haas
242,840,2024,0.384615,stroll,117,aston_martin


DNF rate is a very powerful explanatory variable at least on paper, it is a direct indicator of points and we expect drivers with a high dnf rate to have a low point score. This feature captures both the drivers mistake, and the constructor caused DNFS. 

To capture driver performance beyond car pace, I intend to create a feature called position improvement, defined as the difference between average starting grid position and average finishing position at the season level. This metric acts as a representation of a driver's in-race performance.

Since average grid position was calculated for all races including those in which DNFS were the resulting position, computing this improvement with that feature would introduce bias, which is why I will recalculate so that the position improvement is calculated ONLY in races for which there are NO DNFS, as DNF will already by accounted for in prediction.

### Position delta

In [16]:
con.execute(
"UPDATE driver_season ds "
"SET avg_grid = agg.avg_grid "
"FROM ( "
"    SELECT driverId, year, "
"           AVG(grid) AS avg_grid "
"    FROM results_drivers_races_circuits "
"    WHERE position IS NOT NULL "
"      AND grid IS NOT NULL "
"    GROUP BY driverId, year "
") agg "
"WHERE ds.driverId = agg.driverId "
"AND ds.year = agg.year "
)
con.close()

In [17]:
con = duckdb.connect("f1.duckdb")

con.execute("SELECT (avg_grid) FROM driver_season ").df()

,avg_grid
0,1.705882
1,8.833333
2,8.333333
3,6.588235
4,5.833333
...,...
239,13.416667
240,11.866667
241,9.421053
242,11.750000


In [18]:
con.close()

position_delta = avg_grid - avg_finish. If a driver's position delta is positive then it means that on average, he improves a position_delta number of positions on a race, and if it is negative, it means that on average, he worsens a position_delta number of positions on a race.

In [19]:
con = duckdb.connect("f1.duckdb")
con.execute("ALTER TABLE features "
"ADD COLUMN position_delta double;")

In [20]:
con.execute(
"UPDATE features f "
"SET position_delta = ds.avg_grid - ds.avg_finish "
"FROM driver_season ds "
"WHERE f.driverId = ds.driverId "
"AND f.year = ds.year "
)

In [21]:
con.execute("SELECT (*) FROM features").df()

,driverId,year,DNF_rate,ref,constructor,constructorRef,position_delta
0,3,2014,0.157895,rosberg,131,mercedes,-0.823529
1,825,2014,0.368421,kevin_magnussen,1,mclaren,-0.555556
2,18,2014,0.315789,button,1,mclaren,0.833333
3,4,2014,0.105263,alonso,6,ferrari,1.176471
4,822,2014,0.105263,bottas,3,williams,0.277778
...,...,...,...,...,...,...,...
239,859,2023,0.400000,lawson,213,alphatauri,1.600000
240,20,2020,0.470588,vettel,6,ferrari,1.466667
241,825,2023,0.681818,kevin_magnussen,210,haas,-3.111111
242,840,2024,0.384615,stroll,117,aston_martin,0.583333


In [22]:
con.close()

Driver's with a high DNF rate may have a strong position delta since their results are gonna be based on a few races.

In [23]:
con = duckdb.connect("f1.duckdb")

high_dnf = con.execute(
    "SELECT * FROM features WHERE DNF_rate > 0.30"
).df()

high_dnf

,driverId,year,DNF_rate,ref,constructor,constructorRef,position_delta
0,825,2014,0.368421,kevin_magnussen,1,mclaren,-0.555556
1,18,2014,0.315789,button,1,mclaren,0.833333
2,8,2014,0.368421,raikkonen,6,ferrari,-0.277778
3,818,2014,0.473684,vergne,5,toro_rosso,0.785714
4,826,2014,0.684211,kvyat,5,toro_rosso,0.428571
...,...,...,...,...,...,...,...
165,859,2023,0.400000,lawson,213,alphatauri,1.600000
166,20,2020,0.470588,vettel,6,ferrari,1.466667
167,825,2023,0.681818,kevin_magnussen,210,haas,-3.111111
168,840,2024,0.384615,stroll,117,aston_martin,0.583333


EX: In the 2020 season, vettel had a DNF rate of 47.05%

In [24]:
con.close()

This feature on itself is going to be misleading, but paired with DNF rate will give the model a broader understanding of driver performance, so it's essential to keep this in mind moving forward.

It is very important to have in mind that the position delta for very dominant drivers (such as Hamilton 2014-2020) is expected to be near 0 since they usually start in very high positions and end in the same position, and if there are any changes in their finish position in respect to their grid position it is going to be low, this makes it crucial for this feature to be used alongside other explanatory variables which will give more context to the model. I considered normalizing this feature, but for a xgboost model, or even a linear model with other explanatory variables, this ceiling will be understood by the model.

### High altitude success rate

High altitude is an important factor from a mechanical aspect, so I intend to create a feature that captures how good a driver perform in these high altitude settings which usually represent a more difficult scenario for a driver in a race.

Moving forward I will define high altitude as any altitude that is of 500m AND above.

In [25]:
con = duckdb.connect("f1.duckdb")
high_alt_circuits = con.execute("SELECT * FROM circuits WHERE alt >= 500; ").df()
high_alt_circuits

,circuitId,circuitRef,name,location,country,lat,lng,alt,url
0,16,fuji,Fuji Speedway,Oyama,Japan,35.3717,138.92700,583,http://en.wikipedia.org/wiki/Fuji_Speedway
1,18,interlagos,Autódromo José Carlos Pace,São Paulo,Brazil,-23.7036,-46.69970,785,http://en.wikipedia.org/wiki/Aut%C3%B3dromo_Jo...
2,20,nurburgring,Nürburgring,Nürburg,Germany,50.3356,6.94750,578,http://en.wikipedia.org/wiki/N%C3%BCrburgring
3,80,vegas,Las Vegas Strip Street Circuit,Las Vegas,United States,36.1147,-115.17300,642,https://en.wikipedia.org/wiki/Las_Vegas_Grand_...
4,30,kyalami,Kyalami,Midrand,South Africa,-25.9894,28.07670,1460,http://en.wikipedia.org/wiki/Kyalami
5,32,rodriguez,Autódromo Hermanos Rodríguez,Mexico City,Mexico,19.4042,-99.09070,2227,http://en.wikipedia.org/wiki/Aut%C3%B3dromo_He...
6,36,jacarepagua,Autódromo Internacional Nelson Piquet,Rio de Janeiro,Brazil,-22.9756,-43.39500,1126,http://en.wikipedia.org/wiki/Aut%C3%B3dromo_In...
7,44,las_vegas,Las Vegas Street Circuit,Nevada,USA,36.1162,-115.17400,639,http://en.wikipedia.org/wiki/Las_Vegas_Street_...
8,45,jarama,Jarama,Madrid,Spain,40.6171,-3.58558,609,http://en.wikipedia.org/wiki/Circuito_Permanen...
9,51,charade,Charade Circuit,Clermont-Ferrand,France,45.7472,3.03889,790,http://en.wikipedia.org/wiki/Charade_Circuit


In [26]:
#High altitude race results per driver
con.execute(
"SELECT r.driverId, r.year, r.circuitId, r.position, r.grid "
"FROM results_drivers_races_circuits r "
"JOIN circuits c "
  "ON r.circuitId = c.circuitId "
"WHERE c.alt >= 500 AND r.year >= 2014;").df()

,driverId,year,circuitId,position,grid
0,3,2014,18,1.0,1
1,1,2014,18,2.0,2
2,13,2014,18,3.0,3
3,18,2014,18,4.0,5
4,20,2014,18,5.0,6
...,...,...,...,...,...
641,840,2021,18,NaN,14
642,842,2021,18,7.0,7
643,839,2021,18,8.0,8
644,852,2021,18,15.0,15


In [27]:
con.execute("ALTER TABLE features "
"ADD COLUMN high_alt_sucess_rate double;")

High altitude success rate is defined as the proportion of high altitude races in which a driver finished at his average finish position OR higher

In [28]:
# High-altitude success rate per driver-season
high_alt_success_rate = con.execute(
"SELECT "
"    r.driverId, "
"    r.year, "
"    CAST(SUM(CASE WHEN r.position IS NOT NULL AND r.position <= ds.avg_finish THEN 1 ELSE 0 END) AS FLOAT) / COUNT(r.position) AS high_alt_success_rate "
"FROM results_drivers_races_circuits r "
"JOIN circuits c "
"    ON r.circuitId = c.circuitId "
"JOIN driver_season ds "
"    ON r.driverId = ds.driverId AND r.year = ds.year "
"WHERE r.year >= 2014 AND c.alt >= 500 "
"GROUP BY r.driverId, r.year;"
).df()
high_alt_success_rate

,driverId,year,high_alt_success_rate
0,8,2016,0.500000
1,3,2016,0.666667
2,817,2016,0.333333
3,18,2016,0.333333
4,832,2016,0.666667
...,...,...,...
225,829,2015,0.500000
226,20,2020,0.500000
227,857,2023,0.250000
228,154,2014,0.000000


In [29]:
con.execute(
"UPDATE features f "
"SET high_alt_sucess_rate = sub.high_alt_sucess_rate "
"FROM ( "
"    SELECT r.driverId, r.year, "
"           AVG(CASE "
"               WHEN r.position IS NOT NULL AND r.position <= ds.avg_finish THEN 1.0 "
"               ELSE 0.0 "
"           END) AS high_alt_sucess_rate "
"    FROM results_drivers_races_circuits r "
"    JOIN circuits c "
"      ON r.circuitId = c.circuitId "
"    JOIN driver_season ds "
"      ON r.driverId = ds.driverId AND r.year = ds.year "
"    WHERE r.year >= 2014 AND c.alt >= 500 "
"    GROUP BY r.driverId, r.year "
") sub "
"WHERE f.driverId = sub.driverId AND f.year = sub.year;"
)


In [30]:
con.execute("SELECT (*) FROM features LIMIT 5").df()

,driverId,year,DNF_rate,ref,constructor,constructorRef,position_delta,high_alt_sucess_rate
0,3,2014,0.157895,rosberg,131,mercedes,-0.823529,1.0
1,825,2014,0.368421,kevin_magnussen,1,mclaren,-0.555556,1.0
2,18,2014,0.315789,button,1,mclaren,0.833333,0.5
3,4,2014,0.105263,alonso,6,ferrari,1.176471,0.5
4,822,2014,0.105263,bottas,3,williams,0.277778,0.5


EX: In the 2014 season, Bottas had a 50% success rate in high altitude tracks.

In [31]:
con.close()

Now we have a couple strong driver-oriented performance metrics engineered, due to the limitation of the data at hand, I will now move on to team-oriented performance metrics, although I may come back in the future to work on this.

## Team 
- Team average position

Constructors are the most important and determinant factor for scoring points in a race. Team finish rate is directly correlated to point scoring, along with average finish position as an indicator of car pace, since it includes both drivers of the same car, a delta of the grid and finish will also show how effective their strategy is. Moving forward with all the above mentioned features, I would maybe create a feature that compares how much the driver overperforms their team average, which will really shine light into the performance of a driver.

In [32]:
con = duckdb.connect("f1.duckdb")
con.execute("SELECT * FROM constructors LIMIT 2").df()

,constructorId,constructorRef,name,nationality,url
0,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren
1,2,bmw_sauber,BMW Sauber,German,http://en.wikipedia.org/wiki/BMW_Sauber


In [33]:
con.execute("SELECT * FROM results LIMIT 2").df()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1


In [34]:
con.execute("SELECT * FROM races LIMIT 2;").df()

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
0,1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1,2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N


### Team average finish position

To create a team average finish posiition, I will filter results by race, and then filter the finish positions to the constructor id, that way all of the results of a constructor per race will be available. To do this, I will use the table previously created results_drivers_races_circuits, which contains all the necessary id's.

In [35]:
con.execute("ALTER TABLE features ADD COLUMN team_avg_position double;")
con.execute("ALTER TABLE results_drivers_races_circuits ADD COLUMN team_avg_position double;")

In [36]:
con.execute("SELECT * FROM results_drivers_races_circuits WHERE year >= 2014 AND constructorId = 6 LIMIT 6").df()

,resultId,raceId,driverId,constructorId,grid,position,positionOrder,points,laps,milliseconds,...,driverRef,dob,year,round,circuitId,circuitRef,lat,lng,alt,team_avg_position
0,22133,900,4,6,5,4,4,12.0,57,5613994.0,...,alonso,1981-07-29,2014,1,1,albert_park,-37.84970,144.9680,10,NaN
1,22136,900,8,6,11,7,7,6.0,57,5636385.0,...,raikkonen,1979-10-17,2014,1,1,albert_park,-37.84970,144.9680,10,NaN
2,22155,901,4,6,4,4,4,12.0,56,6061966.0,...,alonso,1981-07-29,2014,2,2,sepang,2.76083,101.7380,18,NaN
3,22163,901,8,6,6,12,12,0.0,55,NaN,...,raikkonen,1979-10-17,2014,2,2,sepang,2.76083,101.7380,18,NaN
4,22182,902,4,6,9,9,9,2.0,57,6015338.0,...,alonso,1981-07-29,2014,3,3,bahrain,26.03250,50.5106,7,NaN
5,22183,902,8,6,5,10,10,1.0,57,6016205.0,...,raikkonen,1979-10-17,2014,3,3,bahrain,26.03250,50.5106,7,NaN


In [37]:
con.execute(
"UPDATE results_drivers_races_circuits r "
"SET team_avg_position  = t.avg_finish_xrace "
"FROM ( "
"    SELECT "
"        raceId, "
"        constructorId, "
"        AVG(position) AS avg_finish_xrace "
"    FROM results_drivers_races_circuits "
"    WHERE position IS NOT NULL "
"      AND year >= 2014 "
"    GROUP BY raceId, constructorId "
") t "
"WHERE r.raceId = t.raceId "
"  AND r.constructorId = t.constructorId;"
)
con.close()

In [38]:
con = duckdb.connect("f1.duckdb")

con.execute("SELECT * FROM results_drivers_races_circuits WHERE year>=2014 AND constructorId = 6 LIMIT 6;").df()


,resultId,raceId,driverId,constructorId,grid,position,positionOrder,points,laps,milliseconds,...,driverRef,dob,year,round,circuitId,circuitRef,lat,lng,alt,team_avg_position
0,22133,900,4,6,5,4,4,12.0,57,5613994.0,...,alonso,1981-07-29,2014,1,1,albert_park,-37.84970,144.9680,10,5.5
1,22136,900,8,6,11,7,7,6.0,57,5636385.0,...,raikkonen,1979-10-17,2014,1,1,albert_park,-37.84970,144.9680,10,5.5
2,22155,901,4,6,4,4,4,12.0,56,6061966.0,...,alonso,1981-07-29,2014,2,2,sepang,2.76083,101.7380,18,8.0
3,22163,901,8,6,6,12,12,0.0,55,NaN,...,raikkonen,1979-10-17,2014,2,2,sepang,2.76083,101.7380,18,8.0
4,22182,902,4,6,9,9,9,2.0,57,6015338.0,...,alonso,1981-07-29,2014,3,3,bahrain,26.03250,50.5106,7,9.5
5,22183,902,8,6,5,10,10,1.0,57,6016205.0,...,raikkonen,1979-10-17,2014,3,3,bahrain,26.03250,50.5106,7,9.5


In [39]:
con.execute("SELECT * FROM results_drivers_races_circuits WHERE year>=2014 AND constructorId = 131 LIMIT 6;").df()

,resultId,raceId,driverId,constructorId,grid,position,positionOrder,points,laps,milliseconds,...,driverRef,dob,year,round,circuitId,circuitRef,lat,lng,alt,team_avg_position
0,22130,900,3,131,3,1.0,1,25.0,57,5578710.0,...,rosberg,1985-06-27,2014,1,1,albert_park,-37.84970,144.9680,10,1.0
1,22148,900,1,131,1,NaN,19,0.0,2,NaN,...,hamilton,1985-01-07,2014,1,1,albert_park,-37.84970,144.9680,10,1.0
2,22152,901,1,131,1,1.0,1,25.0,56,6025974.0,...,hamilton,1985-01-07,2014,2,2,sepang,2.76083,101.7380,18,1.5
3,22153,901,3,131,3,2.0,2,18.0,56,6043287.0,...,rosberg,1985-06-27,2014,2,2,sepang,2.76083,101.7380,18,1.5
4,22174,902,1,131,2,1.0,1,25.0,57,5982743.0,...,hamilton,1985-01-07,2014,3,3,bahrain,26.03250,50.5106,7,1.5
5,22175,902,3,131,1,2.0,2,18.0,57,5983828.0,...,rosberg,1985-06-27,2014,3,3,bahrain,26.03250,50.5106,7,1.5


team avg position per race was effectively created, now we can aggregate it to each driver's season, to do that, we first have to convert the average position per race into average position per season, and then link it to each driver-season.

In [40]:
con.execute("SELECT AVG(team_avg_position) AS season_avg_finish, constructorId, year " 
"FROM results_drivers_races_circuits "
"WHERE year >= 2014 AND team_avg_position IS NOT NULL "
"GROUP BY constructorId, year "
"ORDER BY year, constructorId  ").df()

,season_avg_finish,constructorId,year
0,8.710526,1,2014
1,6.157895,3,2014
2,11.088235,5,2014
3,7.473684,6,2014
4,4.444444,9,2014
...,...,...,...
107,10.307692,117,2024
108,6.269231,131,2024
109,11.791667,210,2024
110,13.269231,214,2024


In [41]:
# Add team avg finish in a season to features 
con.execute("UPDATE features f "
"SET team_avg_position = t.season_avg_finish "
"FROM ( "
    "SELECT AVG(team_avg_position) AS season_avg_finish, constructorId, year " 
    "FROM results_drivers_races_circuits "
    "WHERE year >= 2014 AND team_avg_position IS NOT NULL "
    "GROUP BY constructorId, year ) t "            
"WHERE f.constructor = t.constructorId "
"  AND f.year = t.year;"
)


In [42]:
con.execute("SELECT * FROM features").df()

,driverId,year,DNF_rate,ref,constructor,constructorRef,position_delta,high_alt_sucess_rate,team_avg_position
0,3,2014,0.157895,rosberg,131,mercedes,-0.823529,1.000000,1.921053
1,825,2014,0.368421,kevin_magnussen,1,mclaren,-0.555556,1.000000,8.710526
2,18,2014,0.315789,button,1,mclaren,0.833333,0.500000,8.710526
3,4,2014,0.105263,alonso,6,ferrari,1.176471,0.500000,7.473684
4,822,2014,0.105263,bottas,3,williams,0.277778,0.500000,6.157895
...,...,...,...,...,...,...,...,...,...
239,859,2023,0.400000,lawson,213,alphatauri,1.600000,NaN,12.954545
240,20,2020,0.470588,vettel,6,ferrari,1.466667,0.333333,8.866667
241,825,2023,0.681818,kevin_magnussen,210,haas,-3.111111,0.250000,14.568182
242,840,2024,0.384615,stroll,117,aston_martin,0.583333,0.000000,10.307692


## Driver consistency 

- Driver vs team
- Finish volatility
- Top 10 rate
- Average finish position
- Driver experience

### DVT

From team avg finish I will derive a driver vs team metric, which measures the over or under performance of a driver in respect to its team. 

driver_vs_team = - (team_avg_position - driver_avg_finish ) / team_avg_position

a negative dvt (driver_vs_team) means the driver is undeperforming, and a positive means a driver is overperforming. This feature is a clear signal of a driver's skills in respect to its team.

In [43]:
con.execute("SELECT * FROM driver_season LIMIT 3").df()

,driverId,year,total_points,races_entered,dnfs,avg_grid,avg_finish,completed_races,ref,constructor
0,3,2014,317.0,19,2,1.705882,2.529412,16,rosberg,131
1,825,2014,55.0,19,1,8.833333,9.388889,12,kevin_magnussen,1
2,18,2014,126.0,19,1,8.333333,7.500000,13,button,1


In [44]:
con.execute("ALTER TABLE features ADD COLUMN dvt double;")

In [45]:
con.execute(
"UPDATE features f "
"SET dvt = (f.team_avg_position - ds.avg_finish) / f.team_avg_position "
"FROM driver_season ds "
"WHERE ds.driverId = f.driverId "
    "AND ds.year = f.year "
    "AND ds.avg_finish IS NOT NULL "
    "AND f.team_avg_position IS NOT NULL;"
)

In [46]:
con.execute("SELECT * FROM features LIMIT 5;").df()

,driverId,year,DNF_rate,ref,constructor,constructorRef,position_delta,high_alt_sucess_rate,team_avg_position,dvt
0,3,2014,0.157895,rosberg,131,mercedes,-0.823529,1.0,1.921053,-0.316680
1,825,2014,0.368421,kevin_magnussen,1,mclaren,-0.555556,1.0,8.710526,-0.077878
2,18,2014,0.315789,button,1,mclaren,0.833333,0.5,8.710526,0.138973
3,4,2014,0.105263,alonso,6,ferrari,1.176471,0.5,7.473684,0.275891
4,822,2014,0.105263,bottas,3,williams,0.277778,0.5,6.157895,0.097816


In [47]:
con.close()

In [48]:
con = duckdb.connect("f1.duckdb")
con.execute("SELECT * FROM features WHERE driverId = 4;").df()

,driverId,year,DNF_rate,ref,constructor,constructorRef,position_delta,high_alt_sucess_rate,team_avg_position,dvt
0,4,2014,0.105263,alonso,6,ferrari,1.176471,0.500000,7.473684,0.275891
1,4,2015,0.833333,alonso,1,mclaren,3.818182,0.000000,12.033333,-0.012339
2,4,2016,0.600000,alonso,1,mclaren,2.117647,0.333333,10.404762,0.021941
3,4,2017,0.842105,alonso,1,mclaren,1.909091,0.666667,11.694444,0.043835
4,4,2018,0.761905,alonso,1,mclaren,2.933333,0.333333,11.476190,0.111203
5,4,2021,0.500000,alonso,214,alpine,1.100000,0.750000,9.142857,0.010156
6,4,2022,0.454545,alonso,214,alpine,1.352941,0.333333,8.428571,-0.025922
7,4,2023,0.090909,alonso,117,aston_martin,0.400000,0.500000,7.545455,0.257831
8,4,2024,0.307692,alonso,117,aston_martin,-0.307692,0.000000,10.307692,0.037313


EX: In the 2023 season, Alonso overperformed his teams baseline finish position by 25.78% 

### Finish volatility

Finish volatility is a feature meant to measure the dispersity of a driver's race results throughout a season. It is calculated as the square root of the square difference (STDEV) between a driver's finish position and their average finish position, divided over all the races in which they finished. This feature adds another perspective to the point scoring goal, a driver who has a very low volatility is a driver who scores around the same points every race, while a driver with high volatility is one that is not reliable. This feature on itself is more meaningful than average finish.

In [49]:
con.execute("ALTER TABLE features ADD COLUMN finish_volatility double;")

In [50]:
con.execute("UPDATE features f "
"SET finish_volatility = fv.finish_volatility "
"FROM ("
    "SELECT driverId, year, "
    "STDDEV_POP(position) AS finish_volatility "
    "FROM results_drivers_races_circuits "
    "WHERE position IS NOT NULL AND year >= 2014 "
    "GROUP BY driverId, year ) fv "
"WHERE f.driverId = fv.driverId "
"AND f.year = fv.year;")

In [51]:
con.execute("SELECT ref, constructorRef, finish_volatility FROM features WHERE year = 2023 AND constructor = 9 LIMIT 9;").df()

,ref,constructorRef,finish_volatility
0,perez,red_bull,3.442383
1,max_verstappen,red_bull,0.862439


EX: Max's vestappen dominant 2023 season showcases in how little volatility (0.86) he had that season (21 out of 22 podiums). While Checo Perez's higher volatility (3.44) is coherent with his bumpy season.

It is important to note that the unit for this variable is POSITIONS, so the 0.86 for verstappen's 2023 season would mean he had a volatility of 0.86 positions.

In [52]:
con.close()

### Driver experience

Our next feature of interest is the experience of the driver. To do this, I will calculate a driver's experience as
first season - year of current season. 
First season will be calculated from 2014 onwards. Even though this is not the proper definition of experience, driver's with many years of experience will have a very high number of years active, while a new driver with a few years of experience is going to have a low number of years active, which would make this statistic biased, this is conterintuitive in respect to the goal of 2014 and after era.

In [53]:
con = duckdb.connect("f1.duckdb")
con.execute("SELECT * FROM results_drivers_races_circuits LIMIT 1;").df()

,resultId,raceId,driverId,constructorId,grid,position,positionOrder,points,laps,milliseconds,...,driverRef,dob,year,round,circuitId,circuitRef,lat,lng,alt,team_avg_position
0,1,18,1,1,1,1,1,10.0,58,5690616,...,hamilton,1985-01-07,2008,1,1,albert_park,-37.8497,144.968,10,NaN


In [54]:
con.execute(
    "SELECT driverId, driverRef, MIN(year) AS first_year, "
    "FROM results_drivers_races_circuits "
    "WHERE year >= 2014 AND driverId = 1 "
    "GROUP BY driverId, driverRef, year;").df()

,driverId,driverRef,first_year
0,1,hamilton,2014
1,1,hamilton,2015
2,1,hamilton,2016
3,1,hamilton,2017
4,1,hamilton,2018
5,1,hamilton,2019
6,1,hamilton,2020
7,1,hamilton,2021
8,1,hamilton,2022
9,1,hamilton,2023


In [55]:
con.execute("ALTER TABLE features ADD COLUMN years_active BIGINT;")

In [56]:
con.execute("UPDATE features f "
"SET years_active = da.years_active "
    "FROM ( "
    "SELECT driverId, year, "
    "year - MIN(year) OVER (PARTITION BY driverId) AS years_active "        
    "FROM results_drivers_races_circuits "
    "WHERE year >= 2014 "
    "GROUP BY driverId, year ) da "
"WHERE f.driverId = da.driverId "
"AND f.year = da.year"            
           )

In [57]:
con.execute(
    "SELECT ref, year, years_active "
    "FROM features WHERE ref = 'hamilton'").df()

,ref,year,years_active
0,hamilton,2014,0
1,hamilton,2015,1
2,hamilton,2016,2
3,hamilton,2017,3
4,hamilton,2018,4
5,hamilton,2019,5
6,hamilton,2020,6
7,hamilton,2021,7
8,hamilton,2022,8
9,hamilton,2023,9


In [58]:
con.close()

### Top 10 rate

Top 10 rate measures how consistently a driver converts race starts into points finishes within a season. In contrasat to average finish, it focuses on scoring reliability rather than a tendency.

In [59]:
con = duckdb.connect("f1.duckdb")
con.execute(
    "SELECT driverRef, year, position "
    "FROM results_drivers_races_circuits "
    "WHERE year >= 2014 AND position <= 10;").df()

,driverRef,year,position
0,rosberg,2014,1
1,kevin_magnussen,2014,2
2,button,2014,3
3,alonso,2014,4
4,bottas,2014,5
...,...,...,...
2165,hamilton,2022,8
2166,bottas,2022,9
2167,vettel,2022,10
2168,gasly,2021,7


In [60]:
con.execute("ALTER TABLE features ADD COLUMN top10 DOUBLE;")

In [61]:
con.execute(
"UPDATE features f "
"SET top10 = t.top10 "
"FROM ( "
    "SELECT "
        "driverId, "
        "year, "
        "AVG(CASE WHEN position <= 10 THEN 1.0 ELSE 0.0 END) AS top10 "
    "FROM results_drivers_races_circuits "
    "WHERE position IS NOT NULL "
      "AND year >= 2014 "
    "GROUP BY driverId, year "
") t "
"WHERE f.driverId = t.driverId "
"AND f.year = t.year;"
)


In [62]:
con.execute("SELECT ref, year, top10 FROM features  WHERE ref='alonso';").df()

,ref,year,top10
0,alonso,2014,1.000000
1,alonso,2015,0.181818
2,alonso,2016,0.529412
3,alonso,2017,0.454545
4,alonso,2018,0.600000
5,alonso,2021,0.750000
6,alonso,2022,0.823529
7,alonso,2023,0.950000
8,alonso,2024,0.615385


In [63]:
con.execute("SELECT ref, year, top10 FROM features  WHERE ref='grosjean';").df()

,ref,year,top10
0,grosjean,2015,0.769231
1,grosjean,2016,0.312500
2,grosjean,2017,0.470588
3,grosjean,2018,0.466667
4,grosjean,2019,0.200000
5,grosjean,2014,0.153846
6,grosjean,2020,0.083333


In [64]:
con.execute("DESCRIBE features").df()

,column_name,column_type,null,key,default,extra
0,driverId,BIGINT,YES,NaN,NaN,NaN
1,year,BIGINT,YES,NaN,NaN,NaN
2,DNF_rate,DOUBLE,YES,NaN,NaN,NaN
3,ref,VARCHAR,YES,NaN,NaN,NaN
4,constructor,VARCHAR,YES,NaN,NaN,NaN
5,constructorRef,VARCHAR,YES,NaN,NaN,NaN
6,position_delta,DOUBLE,YES,NaN,NaN,NaN
7,high_alt_sucess_rate,DOUBLE,YES,NaN,NaN,NaN
8,team_avg_position,DOUBLE,YES,NaN,NaN,NaN
9,dvt,DOUBLE,YES,NaN,NaN,NaN


In [65]:
con.close()

### Average finish position

This feature is already created in the driver_season as a DOUBLE type, it acts as a strong performance baseline that summarizes roughly season level performance as a easily interpretable metric.   

In [66]:
con = duckdb.connect("f1.duckdb")

con.execute("SELECT * FROM driver_season WHERE ref='bottas'").df()

,driverId,year,total_points,races_entered,dnfs,avg_grid,avg_finish,completed_races,ref,constructor
0,822,2014,186.0,19,1,5.833333,5.555556,17,bottas,3
1,822,2015,136.0,19,2,5.823529,6.764706,14,bottas,3
2,822,2016,85.0,21,2,8.052632,8.526316,12,bottas,3
3,822,2017,305.0,20,1,3.368421,2.947368,19,bottas,131
4,822,2018,247.0,21,2,4.368421,3.894737,18,bottas,131
5,822,2019,326.0,21,2,3.736842,2.684211,18,bottas,131
6,822,2020,223.0,17,1,2.375000,4.312500,15,bottas,131
7,822,2021,219.0,22,4,5.777778,5.000000,17,bottas,131
8,822,2022,47.0,22,5,10.235294,10.411765,11,bottas,51
9,822,2023,10.0,22,3,12.421053,13.368421,12,bottas,51


In [67]:
con.execute("ALTER TABLE features ADD COLUMN avg_finish DOUBLE;")

In [68]:
con.execute("UPDATE features f "
"SET avg_finish = t.avg_finish "
"FROM ("
    "SELECT driverId, avg_finish, year "
    "FROM driver_season) t "
"WHERE f.driverId = t.driverId "
"AND f.year = t.year;"          
 )

In [69]:
con.execute("SELECT * FROM features LIMIT 5;").df()

,driverId,year,DNF_rate,ref,constructor,constructorRef,position_delta,high_alt_sucess_rate,team_avg_position,dvt,finish_volatility,years_active,top10,avg_finish
0,3,2014,0.157895,rosberg,131,mercedes,-0.823529,1.0,1.921053,-0.316680,2.952918,0,0.941176,2.529412
1,825,2014,0.368421,kevin_magnussen,1,mclaren,-0.555556,1.0,8.710526,-0.077878,2.850709,0,0.666667,9.388889
2,18,2014,0.315789,button,1,mclaren,0.833333,0.5,8.710526,0.138973,3.670453,0,0.722222,7.500000
3,4,2014,0.105263,alonso,6,ferrari,1.176471,0.5,7.473684,0.275891,1.816876,0,1.000000,5.411765
4,822,2014,0.105263,bottas,3,williams,0.277778,0.5,6.157895,0.097816,2.650413,0,0.944444,5.555556


In [70]:
con.execute("SELECT ref, year, avg_finish FROM features where ref = 'bottas' ORDER BY year").df()

,ref,year,avg_finish
0,bottas,2014,5.555556
1,bottas,2015,6.764706
2,bottas,2016,8.526316
3,bottas,2017,2.947368
4,bottas,2018,3.894737
5,bottas,2019,2.684211
6,bottas,2020,4.312500
7,bottas,2021,5.000000
8,bottas,2022,10.411765
9,bottas,2023,13.368421


In [71]:
con.close()

## Features Summary

I feel confident that this set of driver-focused features provides a solid foundation to approximate point scored in a season for a driver given the constraints in the accessible data.

In [72]:
con = duckdb.connect("f1.duckdb")
con.execute("DESCRIBE features").df()

,column_name,column_type,null,key,default,extra
0,driverId,BIGINT,YES,NaN,NaN,NaN
1,year,BIGINT,YES,NaN,NaN,NaN
2,DNF_rate,DOUBLE,YES,NaN,NaN,NaN
3,ref,VARCHAR,YES,NaN,NaN,NaN
4,constructor,VARCHAR,YES,NaN,NaN,NaN
5,constructorRef,VARCHAR,YES,NaN,NaN,NaN
6,position_delta,DOUBLE,YES,NaN,NaN,NaN
7,high_alt_sucess_rate,DOUBLE,YES,NaN,NaN,NaN
8,team_avg_position,DOUBLE,YES,NaN,NaN,NaN
9,dvt,DOUBLE,YES,NaN,NaN,NaN


- driverId: Driver identifier accross all tables and raw datasets.
- year: Season year.
- ref: Driver's name reference.
- constructor: Team's identifier accross all tables and raw datasets.
- constructorRef: Team's name reference.
- DNF_rate: Proportion of races in which a driver did not finish in a season.
- position_delta: Average finish in relation to starting position, measurement of reliability.
- high_alt_success_rate: Proportion of races on high altitude tracks in a season in which a driver finished at or better than their average position, measurement of skill
- team_avg_position: Driver's team's average finish position in a season.
- dvt: Normalized measure of a driver's over or underperformance in relation to their teammate.
- finish_volatility: Standard deviation of a driver's finish position in a season. Measurement of consistency.
- years_active: Number of years a driver has of activeness starting from the 2014 season.
- top10: Proportion of races in which a driver finished in the top 10 in a season. Measurement of consistency.
- avg_finish: Average finish position of a driver in a season. 

In [73]:
con.close()

In [74]:
con = duckdb.connect("f1.duckdb")

In [75]:
con.execute("COPY features TO 'features.csv' (HEADER, DELIMITER ',');").df()

,Count
0,244


In [76]:
con.close()

In [77]:
con = duckdb.connect("f1.duckdb")

tables = [
    'driver_season',
    'results_drivers',
    'results_drivers_races',
    'results_drivers_races_circuits',
    'features'
]

for t in tables:
    con.execute(f"COPY {t} TO 'modeling_tables/{t}.csv' (HEADER, DELIMITER ',')")

In [78]:
con.close()